In [ ]:
import boto3, botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile, uuid
from datetime import date
from misc import load_from_yaml, save_to_yaml
# import iam, s3, lf, rds, vpc, ec2

boto3.setup_default_session(profile_name="AMShah")

In [ ]:
from dotenv import load_dotenv
load_dotenv("../env")
AWS_ALL_IN_ONE_SG = os.environ["AWS_ALL_IN_ONE_SG"]
ACCOUNT_ID        = os.environ['AWS_ACCOUNT_ID_ROOT']
REGION            = os.environ['AWS_DEFAULT_REGION']
VPC_ID            = os.environ['AWS_DEFAULT_VPC']
SECURITY_GROUP_ID = os.environ['AWS_DEFAULT_SG_ID']
SUBNET_IDS        = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID         = SUBNET_IDS[0]
AWS_INSTANCE_ID_JMASTER   = os.environ['AWS_INSTANCE_ID_JMASTER']
AWS_INSTANCE_ID_JAGENT    = os.environ['AWS_INSTANCE_ID_JAGENT']
AWS_DEFAULT_IMAGE_ID      = os.environ['AWS_DEFAULT_IMAGE_ID']
AWS_DEFAULT_KEY_PAIR_NAME = os.environ['AWS_DEFAULT_KEY_PAIR_NAME']
AWS_DEFAULT_INSTANCE_TYPE = os.environ['AWS_DEFAULT_INSTANCE_TYPE']
AWS_DEFAULT_IMAGE_ID = os.environ['AWS_DEFAULT_IMAGE_ID']
AMAZON_LINUX_AMI_ID = os.environ["AMAZON_LINUX_AMI_ID"]

In [5]:
sts_client           = boto3.client('sts')
rds_client           = boto3.client('rds')
iam_client           = boto3.client('iam')
s3_client            = boto3.client('s3')
glue_client          = boto3.client('glue')
lakeformation_client = boto3.client('lakeformation')
stepfunctions_client = boto3.client('stepfunctions')
apigateway_client    = boto3.client('apigateway')
lsn_client           = boto3.client('lambda')
events_client        = boto3.client('events')
sqs_client           = boto3.client('sqs')

emr_client = boto3.client('emr', region_name=REGION)

In [6]:
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
elbv2_client = boto3.client("elbv2", region_name=REGION)
autoscaling_client = boto3.client("autoscaling", region_name=REGION)
route53_client = boto3.client("route53", region_name=REGION)
acm_client = boto3.client("acm", region_name=REGION)  # Change region as needed

# CloudFormation is a global service, but you can specify a region for the client
cf_client = boto3.client("cloudformation", region_name=REGION)  

# # Example: Get a specific VPC
# vpc = ec2_resource.Vpc('vpc_id')

# # Example: Get a specific EBS volume
# volume = ec2_resource.Volume('volume_id')

-   [boto3 doc: Route53](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/route53.html) | 


- [Custom Domain Name with AWS API Gateway | Step by Step Tutorial](https://www.youtube.com/watch?v=ESei6XQ7dMg)
  - [Introducing mutual TLS authentication for Amazon API Gateway](https://aws.amazon.com/blogs/compute/introducing-mutual-tls-authentication-for-amazon-api-gateway/)
- [Custom Domain Name with AWS API Gateway | Step by Step Tutorial](https://www.youtube.com/watch?v=ESei6XQ7dMg)
- [How to transfer a domain into AWS Route 53](https://www.youtube.com/watch?v=jzzR-JOoK3E&t=49s)

#### Rout53 Operations

In [ ]:
# Hosted zone parameters
domain_name = "harnesstechtx.com"  # Replace with your actual domain (must end with a dot if it's a fully qualified domain)
caller_reference = str(uuid.uuid4())  # A unique string to ensure the request is idempotent
comment = "Hosted zone for harnesstechtx.com"

# Create the hosted zone
response = route53_client.create_hosted_zone(
    Name=domain_name,
    CallerReference=caller_reference,
    HostedZoneConfig={
        "Comment": comment,
        "PrivateZone": False,  # Set to True if this is a private hosted zone for a VPC
    },
)

# Print the response
print("Hosted Zone Created:")
print("ID:", response["HostedZone"]["Id"])
print("Name:", response["HostedZone"]["Name"])
print("Caller Reference:", response["HostedZone"]["CallerReference"])

In [8]:
zones = route53_client.list_hosted_zones()["HostedZones"]
for zone in zones:
    if zone["Name"] == "harnesstechtx.com.": hosted_zone_id = zone["Id"].split("/")[-1]
print(hosted_zone_id)

Z01482622Y718COFL2WJT


In [ ]:
domain_name = "harnesstechtx.com"
subject_alternative_names = ["www.harnesstechtx.com", "sub.harnesstechtx.com"]

In [ ]:
# List record sets
response = route53_client.list_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    StartRecordName="harnesstechtx.com.",
    StartRecordType="A",
    MaxItems="1",
)

record_set = response["ResourceRecordSets"][0]
print(record_set)


In [ ]:
ALB_ARN=""

In [ ]:
CanonicalHostedZoneId = elbv2_client.describe_load_balancers(LoadBalancerArns=[ALB_ARN])["LoadBalancers"][0]["CanonicalHostedZoneId"]
print(CanonicalHostedZoneId)
print(hosted_zone_id)

Z35SXDOTRQ7X7K
Z04555692B7PI94BFJEBI


In [ ]:
response = route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Add CNAME record",
        "Changes": [
            {
                "Action": "UPSERT",
                "ResourceRecordSet": {
                    "Name": "www.yourdomain.com.",
                    "Type": "CNAME",
                    "TTL": 300,
                    "ResourceRecords": [{"Value": "example.yourdomain.com."}],
                },
            }
        ],
    },
)

In [ ]:
response = route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Add MX record",
        "Changes": [
            {
                "Action": "UPSERT",
                "ResourceRecordSet": {
                    "Name": "yourdomain.com.",
                    "Type": "MX",
                    "TTL": 300,
                    "ResourceRecords": [
                        {"Value": "10 mail1.yourdomain.com."},
                        {"Value": "20 mail2.yourdomain.com."},
                    ],
                },
            }
        ],
    },
)
print(json.dumps(response, indent=4, default=convert_datetime))

In [ ]:
# Create Route53 record set for ALB
res = route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Add Alias record",
        "Changes": [
            {
                "Action": "UPSERT",
                "ResourceRecordSet": {
                    "Name": "harnesstechtx.com.", # Must end with a dot
                    "Type": "A",
                    "AliasTarget": {
                        "HostedZoneId": CanonicalHostedZoneId,  # ELB or CloudFront Zone ID
                        "DNSName": ALB_DNS,
                        "EvaluateTargetHealth": True,
                    },
                },
            }
        ],
    },
)

print(json.dumps(res, indent=4, default=convert_datetime))

In [ ]:
# Delete the existing A record
res = route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Add Alias record",
        "Changes": [
            {
                "Action": "DELETE",
                "ResourceRecordSet": {
                    "Name": "harnesstechtx.com.",  # Must end with a dot
                    "Type": "A",
                    "AliasTarget": {
                        "HostedZoneId": CanonicalHostedZoneId,  # ELB or CloudFront Zone ID
                        "DNSName": ALB_DNS,
                        "EvaluateTargetHealth": True,
                    },
                },
            }
        ],
    },
)
print(json.dumps(res, indent=4, default=convert_datetime))

{
    "ResponseMetadata": {
        "RequestId": "039ebdb0-c6b5-4a84-a9ce-a03b599d159b",
        "HTTPStatusCode": 200,
        "HTTPHeaders": {
            "x-amzn-requestid": "039ebdb0-c6b5-4a84-a9ce-a03b599d159b",
            "content-type": "text/xml",
            "content-length": "318",
            "date": "Sun, 18 May 2025 00:06:26 GMT"
        },
        "RetryAttempts": 0
    },
    "ChangeInfo": {
        "Id": "/change/C03144583EZ4BVVCWM4VJ",
        "Status": "PENDING",
        "SubmittedAt": "2025-05-18T00:06:26.666000+00:00",
        "Comment": "Add Alias record"
    }
}


### [DNS & Amazon Route 53 Deep dive](https://www.youtube.com/watch?v=94vdYMBcE5Y&list=PLO95rE9ahzRsP_ryXpnAHTbKlQIt_JEyF&index=1)

In [5]:
# ! mkdir ./data/route53

In [4]:
# ! curl -o ./data/route53/free_static_webpage.zip https://www.free-css.com/assets/files/free-css-templates/download/page295/makaan.zip

In [6]:
# ! unzip ./data/route53/free_static_webpage.zip -d ./data/route53/

In [7]:
# ! rm -fr ./data/route53/free_static_webpage.zip

In [ ]:
def create_dns_record(
    zone_id,
    record_name,
    record_type,
    record_value,
    ttl=300,
    routing_policy="simple",
    set_identifier=None,
    weight=None,
    region=None,
    ):
    """
    Creates a DNS record in AWS Route 53 with the specified routing policy.

    :param zone_id: (str) Hosted zone ID.
    :param record_name: (str) Fully qualified domain name (FQDN).
    :param record_type: (str) DNS record type (e.g., "A", "CNAME", "TXT").
    :param record_value: (list) List of values for the record.
    :param ttl: (int) Time to Live (TTL) in seconds.
    :param routing_policy: (str) Routing policy ("simple", "weighted", "latency", "failover", "geolocation").
    :param set_identifier: (str) Unique identifier required for weighted, latency, geolocation policies.
    :param weight: (int) Weight value (required for weighted policy).
    :param region: (str) AWS region (required for latency policy).
    """
    client = boto3.client("route53")

    record_set = {
        "Name": record_name,
        "Type": record_type,
        "TTL": ttl,
        "ResourceRecords": [{"Value": val} for val in record_value],
    }

    # Handle routing policies
    if routing_policy == "weighted":
        if set_identifier is None or weight is None:
            raise ValueError("Weighted routing requires 'set_identifier' and 'weight'")
        record_set["SetIdentifier"] = set_identifier
        record_set["Weight"] = weight

    elif routing_policy == "latency":
        if set_identifier is None or region is None:
            raise ValueError("Latency routing requires 'set_identifier' and 'region'")
        record_set["SetIdentifier"] = set_identifier
        record_set["Region"] = region

    elif routing_policy == "failover":
        if set_identifier is None:
            raise ValueError("Failover routing requires 'set_identifier'")
        record_set["SetIdentifier"] = set_identifier
        record_set["Failover"] = "PRIMARY"  # Change to "SECONDARY" as needed

    elif routing_policy == "geolocation":
        if set_identifier is None:
            raise ValueError("Geolocation routing requires 'set_identifier'")
        record_set["SetIdentifier"] = set_identifier
        record_set["GeoLocation"] = {"CountryCode": "US"}  # Modify as needed

    change_batch = {"Changes": [{"Action": "UPSERT", "ResourceRecordSet": record_set}]}

    response = client.change_resource_record_sets(
        HostedZoneId=zone_id, ChangeBatch=change_batch
    )

    return response

In [ ]:
# # Example Usage:
response = create_dns_record(
    zone_id="ZXXXXXXXXXXXXX",
    record_name="example.mydomain.com",
    record_type="A",
    record_value=["192.168.1.1"],
    ttl=300,
    routing_policy="weighted",
    set_identifier="Instance-1",
    weight=50,
)
print(response)